# Validation and errors

Bad inputs that can be found before calculating are issues on a `ValidationReport`: a code,
a severity, a path, and a message. `validate` runs the checks. `is_valid` and `report` only
read the last report. An object that has never been validated has an empty report, and an
empty report is ok.

A problem that makes an object impossible to construct throws. `SjdError` carries `.code`.
`ConfigurationError` and `ValidationError` are also `ValueError`. `NumericalError` is also
`ArithmeticError`. A formula such as `Black76Call` is not validatable: a bad input throws
at construction. Codes are listed in `docs/issue-codes.md`.

`get_value` assumes validation has passed. The notebooks call `require_valid` first.


In [ ]:
import json
import logging
import sqlite3
from datetime import date

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import sjd
from sjd import book, examples
from sjd.markets import schema, store

sjd.set_log_level("INFO")

market = examples.sample_market()
environment = market["environment"]
anchor = market["anchor"]
settlement = market["forward_settlement"]


## A report

`str(report)` is one `path: code: message` line per issue. `to_dict()` is the same report
as data, and the HTML table is what a notebook shows when a cell ends with the report.
`repr` names the type and the number of issues.


In [ ]:
forward = sjd.EuropeanForward(settlement, -1.0, 1.0, sjd.BuySell.BUY)
strike_report = forward.validate(sjd.RatesEnvironment(anchor))

assert not strike_report.ok
assert len(str(strike_report).splitlines()) == len(strike_report.issues)
assert all(": " in line for line in str(strike_report).splitlines())
assert str(strike_report.issues[0]) == str(strike_report).splitlines()[0]
assert "CONTRACT_STRIKE_NEGATIVE" in str(strike_report)
described = strike_report.to_dict()
assert described["ok"] is False
assert described["issues"][0]["code"] == "CONTRACT_STRIKE_NEGATIVE"
assert described["issues"][0]["severity"] == "ERROR"
assert "european_forward" in described["issues"][0]["path"]
assert "<table>" in strike_report._repr_html_()
assert "ValidationReport" in repr(strike_report)
strike_report


## Errors and warnings

An error fails `ok`. A warning does not. The curve below has its first pillar after the
anchor and allows extrapolation, so both findings are warnings and the curve is still valid.


In [ ]:
flat = sjd.VolatilityCurve(
    anchor,
    [date(2026, 12, 16), date(2027, 9, 16)],
    [examples.FLAT_VOLATILITY, examples.FLAT_VOLATILITY],
    allow_extrapolation=True,
)
warning_report = flat.validate()
assert warning_report.ok
assert {issue.code for issue in warning_report.issues} == {
    "RATE_SOURCE_FLAT_BACKFILL",
    "RATE_SOURCE_EXTRAPOLATION",
}
assert all(issue.severity == sjd.ValidationSeverity.WARNING for issue in warning_report.issues)
assert any(issue.severity == sjd.ValidationSeverity.ERROR for issue in strike_report.issues)
warning_report


## Reading the last report

`is_valid` does not run the checks. A new environment reads as valid because its report is
empty. `validate` then reports that no rates are set, and `is_valid` follows that stored
result. It changes only when `validate` runs again.


In [ ]:
fresh = sjd.RatesEnvironment(anchor)
assert fresh.report.issues == ()
assert fresh.is_valid
empty_report = fresh.validate()
assert not empty_report.ok
assert not fresh.is_valid
assert empty_report.issues[0].code == "RATES_ENVIRONMENT_NO_RATES"
assert fresh.report is not None
assert [issue.code for issue in fresh.report.issues] == ["RATES_ENVIRONMENT_NO_RATES"]

option = sjd.EuropeanOption(
    market["collar_maturity"],
    market["collar_settlement"],
    17.5,
    sjd.PutCall.CALL,
    1.0,
    market["collar_settlement"],
    0.0,
    sjd.BuySell.BUY,
)
assert option.is_valid
assert not option.validate(fresh).ok
assert not option.is_valid
assert option.validate(environment).ok
assert option.is_valid
empty_report


## How a path nests

One broken discount curve is the interpolator at the bottom. The environment, the contract,
and the portfolio each put their own prefix in front of that path.


In [ ]:
broken = sjd.DiscountCurve(anchor, [anchor, settlement], [1.0])
held = sjd.RatesEnvironment(anchor)
held.set_frs(sjd.ForwardCurve(anchor, [anchor, settlement], [examples.SPOT, examples.SPOT]))
held.set_drs(broken)
contract = sjd.EuropeanForward(settlement, 17.0, 1.0, sjd.BuySell.BUY)
portfolio = sjd.Portfolio([contract])

curve_path = broken.validate().issues[0].path
environment_path = held.validate().issues[0].path
contract_paths = [issue.path for issue in contract.validate(held).issues]
nested = ("portfolio", "contracts[0]", "european_forward") + environment_path
portfolio_report = portfolio.validate(held)

assert curve_path == ("discount_curve", "dated_curve", "interpolator")
assert environment_path == ("rates_environment", "discount_rate_source") + curve_path
assert ("european_forward",) + environment_path in contract_paths
assert nested in [issue.path for issue in portfolio_report.issues]
for path in (curve_path, environment_path, contract_paths[0], nested):
    print(" / ".join(path))
portfolio_report


## Stopping

`raise_if_invalid` and `require_valid` raise `ValidationError` when the report has an error.
The exception carries the report. A valid report does not raise.


In [ ]:
try:
    strike_report.raise_if_invalid()
except sjd.ValidationError as error:
    assert error.code == "VALIDATION"
    assert error.report is strike_report or str(error.report) == str(strike_report)
    print(error.code, error.report.issues[0].code)
else:
    raise AssertionError("an error report should raise")

try:
    sjd.require_valid(fresh)
except sjd.ValidationError as error:
    assert isinstance(error, ValueError)
    assert isinstance(error, sjd.SjdError)
    assert not error.report.ok
else:
    raise AssertionError("an empty environment should be refused")

sjd.require_valid(environment)
warning_report.raise_if_invalid()


## The exception family

| Exception | Also a | When |
| --- | --- | --- |
| `ConfigurationError` | `ValueError` | An unknown name, or an input a formula cannot price |
| `ValidationError` | `ValueError` | `require_valid` found an error and will not calculate |
| `NumericalError` | `ArithmeticError` | A solver or a conversion did not succeed |
| `SjdError` | `RuntimeError` | The base type. `.code` is the stable name |
| `BookError` | `ValueError` | The TOML book cannot be loaded |
| `CatalogueError` | `ValueError` | The market catalogue cannot be parsed |
| `QuoteNotAvailable` | `KeyError` | The quote has no such date or currency |
| `PillarNotFound` | `LookupError` | No rates label, or no pillar on the chosen label |
| `RatesLabelError` | `ValueError` | A save would change a rates label's anchor date |
| `SchemaVersionError` | `RuntimeError` | The database was written by another schema |


In [ ]:
from sjd import markets

try:
    sjd.Calendar("Narnia")
except sjd.ConfigurationError as error:
    assert isinstance(error, ValueError)
    assert error.code == "CONFIGURATION"
    print("configuration:", error)
else:
    raise AssertionError("Narnia is not a calendar")

numerical = sjd.NumericalError("solver failed")
assert isinstance(numerical, ArithmeticError)
assert not isinstance(numerical, ValueError)
assert numerical.code == "NUMERICAL"

base = sjd.SjdError("Requirement failed: ready", "REQUIRE")
assert isinstance(base, RuntimeError)
assert base.code == "REQUIRE"

try:
    book.parse_book("this is not toml [[")
except book.BookError as error:
    assert isinstance(error, ValueError)
    print("book:", error)
else:
    raise AssertionError("broken TOML should be a BookError")

try:
    markets.parse_catalogue("this is not = a catalogue [[[")
except markets.CatalogueError as error:
    assert isinstance(error, ValueError)
    print("catalogue:", error)
else:
    raise AssertionError("broken markup should be a CatalogueError")

try:
    markets.spot_date(markets.quote_spec("DATED_BRENT", "USD"), anchor)
except markets.QuoteNotAvailable as error:
    assert isinstance(error, KeyError)
    print("quote:", error)
else:
    raise AssertionError("Dated Brent dates are not a spot lag")

connection = sqlite3.connect(":memory:")
schema.initialize(connection)
try:
    store.spot(connection, date(2020, 1, 2), "XAU", "USD")
except store.PillarNotFound as error:
    assert isinstance(error, LookupError)
    print("pillar:", error)
else:
    raise AssertionError("an empty database has no spot")

store.ensure_label(connection, "close", "2026-09-16T16:00:00", anchor)
try:
    store.ensure_label(connection, "close", "2026-09-16T16:00:00", date(2026, 9, 17))
except store.RatesLabelError as error:
    assert isinstance(error, ValueError)
    print("label:", error)
else:
    raise AssertionError("a rates label keeps the anchor it was saved with")
connection.close()

old = sqlite3.connect(":memory:")
old.executescript(
    "CREATE TABLE schema_info (id INTEGER PRIMARY KEY, version INTEGER NOT NULL);"
    "INSERT INTO schema_info VALUES (1, 1);"
)
try:
    schema.initialize(old)
except schema.SchemaVersionError as error:
    assert isinstance(error, RuntimeError)
    assert "schema version" in str(error)
    print("schema:", error)
else:
    raise AssertionError("an older schema should be refused")
old.close()


## The gate before a simulation

A `PortfolioEvolver` on a deterministic forward reports
`PATH_GENERATOR_FORWARD_NOT_STOCHASTIC`. `simulate` validates the same inputs and raises
`ValidationError` with that report, so the paths are not drawn.


In [ ]:
outright = sjd.EuropeanForward(settlement, 17.0, 1.0, sjd.BuySell.BUY)
small_book = sjd.Portfolio([outright])
sjd.require_valid(small_book, environment)
generator = sjd.SingleFactorEconomicScenarioGenerator(environment)
evolver = sjd.PortfolioEvolver(
    small_book, generator, [anchor, settlement]
)
evolver_report = evolver.validate()
assert not evolver_report.ok
assert any(
    issue.code == "PATH_GENERATOR_FORWARD_NOT_STOCHASTIC" for issue in evolver_report.issues
)

try:
    sjd.simulate(small_book, environment, [anchor, settlement], paths=4, seed=1)
except sjd.ValidationError as error:
    assert any(
        issue.code == "PATH_GENERATOR_FORWARD_NOT_STOCHASTIC" for issue in error.report.issues
    )
    print(error.report.issues[0].code)
else:
    raise AssertionError("a deterministic forward cannot drive a simulation")
evolver_report


## Names that remain

These still work. The current names are the ones the notebooks use.

| Still accepted | Use instead |
| --- | --- |
| `is_ok()` | `is_valid` |
| `error_messages()` | the report's messages |
| `validation_report()` | `report` |
| `validate_report()` | `validate()` |


In [ ]:
sjd.require_valid(option, environment)
assert option.is_ok() == option.is_valid
assert option.validation_report().ok == option.report.ok
assert option.validate_report(environment).ok == option.validate(environment).ok
assert isinstance(option.error_messages(), str)
print(option.error_messages() or "no errors")


## Save and reload

A report does not pickle. `to_dict()` does, through JSON, which is how a failed report is
kept beside a rejected run. The file is in a temporary directory.


In [ ]:
import tempfile
from pathlib import Path

scratch = tempfile.TemporaryDirectory()
path = Path(scratch.name) / "report.json"
path.write_text(json.dumps(strike_report.to_dict()), encoding="utf-8")
restored = json.loads(path.read_text(encoding="utf-8"))

assert restored["ok"] is False
assert restored["issues"][0]["code"] == "CONTRACT_STRIKE_NEGATIVE"
assert restored["issues"][0]["path"] == list(strike_report.issues[0].path)
print("reloaded", path.name)
